# Template 07b: SHAP Feature Plots

**Purpose:** Per-feature detail for the top N features (rank from 07a, test importance):
residual plot (actual vs predicted by feature value) and SHAP range plot (SHAP spread by
feature value), for train and test.

**Inputs:**
- results/07a_shap_importance.csv (run 07a first)
- results/06_shap_{train,test}.parquet, results/05c_predictions_{train,test}.parquet
- data/04c_{train,test}_encoded.parquet

**Outputs:**
- plots/07b/{rank}_{feature}_{dataset}_{residual|shap_range}.png (rank prefix sorts by importance)
- results/07b_residual_tables.csv
- results/07b_summary.yaml (plot list + failures)

Config `shap_analysis.display_plots: false` saves the plots without showing them here.

In [ ]:
config_path = "config/car_coll/v1"

In [ ]:
import os
import sys
import yaml
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display

sys.path.insert(0, str(Path.cwd() / 'lib'))
from utils import setup_notebook_environment
project_root = setup_notebook_environment()
import shap_analysis as sa

print("########################################")
print("# STAGE 07b: SHAP FEATURE PLOTS")
print("########################################")

In [ ]:
pc_id = f"PC{open('current.pc').read().strip()}"
with open(f"{config_path}/config.yaml") as f:
    cfg = yaml.safe_load(f)
output_base = cfg["machines"][pc_id]["paths"]["output_path"]
shap_cfg = cfg.get("shap_analysis", {})
top_n = shap_cfg.get("top_n_features", 40)
results_dir = f"{output_base}/results"
plots_dir = f"{output_base}/plots"
os.makedirs(plots_dir, exist_ok=True)
print(f"Output: {output_base} | top_n={top_n}")

datasets = shap_cfg.get("datasets", ["train", "test"])
display_plots = shap_cfg.get("display_plots", True)
feature_dir = f"{plots_dir}/07b"

In [ ]:
# Top features from 07a (test rank)
imp_file = f"{results_dir}/07a_shap_importance.csv"
if not os.path.exists(imp_file):
    raise FileNotFoundError(f"{imp_file} missing - run stage 07a first")
table = pd.read_csv(imp_file)
features = table.loc[table["test_mean_abs_shap"] > 0, "feature"].head(top_n).tolist()
print(f"Plotting {len(features)} features x {len(datasets)} datasets x 2 plot types")

In [ ]:
# Load only the plotted features (test SHAP is large)
frames = {ds: sa.load_shap_data(output_base, ds, features=features) for ds in datasets}
for ds, (s, d) in frames.items():
    print(f"  {ds}: {len(s):,} rows")

In [ ]:
# Generate plots (old 07b plots removed first so the folder matches this run)
import shutil
shutil.rmtree(feature_dir, ignore_errors=True)
plots, tables, failures = sa.make_feature_plots(
    features, frames, feature_dir,
    resid_cfg=shap_cfg.get("residual_plot", {}),
    range_cfg=shap_cfg.get("shap_range_plot", {}))
if len(tables):
    tables.to_csv(f"{results_dir}/07b_residual_tables.csv", index=False)
print(f"Plots saved: {len(plots)} in {feature_dir}")
print(f"Failures:    {len(failures)}")

In [ ]:
# Show plots grouped by feature (rank order): residual train/test, then SHAP range train/test
if display_plots:
    for (rank, feat), grp in plots.groupby(["rank", "feature"], sort=True):
        print(f"\n{'=' * 70}\n#{rank} {feat}\n{'=' * 70}")
        for p in grp.sort_values(["kind", "dataset"], ascending=[True, False])["path"]:
            display(Image(filename=p))
else:
    print(f"display_plots=false - open the PNGs in {feature_dir}")

In [ ]:
summary = {
    "timestamp": datetime.now().isoformat(),
    "features": features,
    "datasets": datasets,
    "n_plots": int(len(plots)),
    "n_failures": int(len(failures)),
    "failures": failures.to_dict("records"),
    "plots_dir": feature_dir,
}
with open(f"{results_dir}/07b_summary.yaml", "w") as f:
    yaml.safe_dump(summary, f, sort_keys=False)
if len(failures):
    print("FAILED PLOTS:")
    print(failures.to_string(index=False))
print(f"Saved: {results_dir}/07b_summary.yaml")

In [ ]:
print("\n########################################")
print("# STAGE 07b: COMPLETE")
print("########################################")